In [ ]:
import pandas as pd
import json
import seaborn as sns
import matplotlib.pyplot as plt
import transformers
from transformers import BertModel
from transformers import BertTokenizer
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import DataLoader, Dataset
from torch import nn
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, AutoModelWithLMHead

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from transformers import Trainer, TrainingArguments


from transformers import AutoModel
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import gc
from datasets import Dataset as HFDataset, DatasetDict
from accelerate import Accelerator

import pickle
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import os

In [ ]:
def load_dataset(clean=True, mount_drive=False):


    dataset_folder = '../data'

    train_path = os.path.join(dataset_folder, 'train_data.json')
    val_path = os.path.join(dataset_folder, 'val_data.json')
    test_path = os.path.join(dataset_folder, 'test_data.json')


    with open(train_path, "r") as f:
        train = json.load(f)
    with open(val_path, "r") as f:
        val = json.load(f)
    with open(test_path, "r") as f:
        test = json.load(f)

    train_df = pd.DataFrame(train["data"])
    val_df = pd.DataFrame(val["data"])
    test_df = pd.DataFrame(test["data"])

    if clean:
        train_len_before = len(train_df)
        val_len_before = len(val_df)
        test_len_before = len(test_df)

        train_df = train_df.dropna(subset=['text'])
        val_df = val_df.dropna(subset=['text'])
        test_df = test_df.dropna(subset=['text'])

        train_removed = train_len_before - len(train_df)
        val_removed = val_len_before - len(val_df)
        test_removed = test_len_before - len(test_df)

        print(f"Removed {train_removed} rows with NaN values from training dataset")
        print(f"Removed {val_removed} rows with NaN values from validation dataset")
        print(f"Removed {test_removed} rows with NaN values from testing dataset")
        print(f"Total removed: {train_removed + val_removed + test_removed} rows")

    return {'train': train_df, 'val': val_df, 'test': test_df}

In [ ]:
dataset= load_dataset(clean=True)

In [ ]:
LABEL_MAP = {
    'neutral': 0,
    'moderate': 1,
    'high-risk': 2
}

EMOTION_INDEX = {
    0: 'neutral',
    1: 'moderate',
    2: 'high-risk'
}

CLASSES = 3
DATACOLUMN = 'text'
LABELCOLUMN = 'my_label'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def create_batch_tokenized_datasets(max_length=512, model_name="microsoft/deberta-v3-base"):
    dataframes = load_dataset(clean=True)

    hf_datasets = {}
    for split, df in dataframes.items():
        df = df.copy()
        df['label'] = df['my_label'].map(LABEL_MAP)
        hf_datasets[split] = HFDataset.from_pandas(df)

    dataset_dict = DatasetDict(hf_datasets)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            max_length=max_length,
            truncation=True,
            return_tensors="pt"
        )

    dataset_encoded = dataset_dict.map(tokenize, batched=True, batch_size=None)

    dataset_encoded.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )

    print(f"Created batch-tokenized datasets:")
    for split, dataset in dataset_encoded.items():
        print(f"  {split}: {len(dataset)} examples")

    return dataset_encoded

In [ ]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)
num_labels = CLASSES
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)

In [ ]:
dataset_encoded = create_batch_tokenized_datasets(max_length=512, model_name=model_name)

In [ ]:
dataset_encoded

In [ ]:
dataset_encoded.set_format(type=None)

class CustomTorchDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(item['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(item['label'], dtype=torch.long)
        }

train_dataset = CustomTorchDataset(dataset_encoded["train"])
val_dataset = CustomTorchDataset(dataset_encoded["val"])

In [ ]:
dataset_encoded["train"].features

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
WARMUP_RATIO = 0.1

In [ ]:
batch_size = 8
logging_steps = len(dataset["train"]) // batch_size
save_steps = logging_steps
eval_steps = logging_steps
training_args = TrainingArguments(
    output_dir="../models/deberta",
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_ratio=WARMUP_RATIO,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    weight_decay=0.01,
    disable_tqdm=False,
    no_cuda=False,
    use_mps_device=False,
    adam_epsilon=1e-6,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    logging_steps=10,
    report_to="none"
)

In [ ]:
trainer_deberta = Trainer(model=model, args=training_args,
                  compute_metrics=compute_metrics,
                  train_dataset=dataset_encoded["train"],
                  eval_dataset=dataset_encoded["val"])
trainer_deberta.train();

In [ ]:
trainer_deberta.save_model("../models/deberta")
tokenizer.save_pretrained("../models/deberta")

In [ ]:
preds_output_deberta = trainer_deberta.predict(dataset_encoded["test"])
preds_output_deberta.metrics

In [ ]:
y_valid = np.array(dataset_encoded["test"]["label"])
y_preds = np.argmax(preds_output_deberta.predictions, axis=1)
labels = ['neutral', 'moderate', 'high-risk']
cm = confusion_matrix(y_valid, y_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
df_preds_deberta = pd.DataFrame({
    "text": dataset_encoded["test"]["text"],
    "true_label": y_valid,
    "predicted_label_deberta": y_preds

})

In [ ]:
df_preds_deberta.to_csv("../results/deberta_predictions.csv", index=False)

In [ ]:
print('\tClassification Report for DeBERTa:\n\n',classification_report(y_valid,y_preds, target_names = ['neutral', 'moderate', 'high-risk']))

Trying Sentence Transformers (all-MiniLM-L6-v2)

In [ ]:
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer, InputExample, models, losses
import math

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

In [ ]:
from sentence_transformers import SentenceTransformer
model_name='all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

In [ ]:
class ClassificationHead(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super(ClassificationHead, self).__init__()
        self.linear = nn.Linear(embedding_dim, num_classes)

    def forward(self, features):
        x = features['sentence_embedding']
        x = self.linear(x)
        return x

class SentenceTransformerWithHead(nn.Module):
    def __init__(self, transformer, head):
        super(SentenceTransformerWithHead, self).__init__()
        self.transformer = transformer
        self.head = head

    def forward(self, input):
        features = self.transformer(input)
        logits = self.head(features)
        return logits

In [ ]:
num_classes = CLASSES
print('class number:', num_classes)
classification_head = ClassificationHead(model.get_sentence_embedding_dimension(), num_classes)

In [ ]:
def train(dataset, num_epochs=3, batch_size=16, learning_rate=2e-5, model_name='all-MiniLM-L6-v2'):

    save_dir = "../models"
    model_name_dir = os.path.join(save_dir, model_name)

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(model_name_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = SentenceTransformer(model_name)
    model.to(device)

    embedding_dim = model.get_sentence_embedding_dimension()
    classification_head = ClassificationHead(embedding_dim, CLASSES)
    classification_head.to(device)

    model_with_head = SentenceTransformerWithHead(model, classification_head)
    model_with_head.to(device)

    train_sentences = dataset['train'][DATACOLUMN].tolist()
    if dataset['train'][LABELCOLUMN].dtype == 'object':
        train_labels = [LABEL_MAP[label] for label in dataset['train'][LABELCOLUMN]]
    else:
        train_labels = dataset['train'][LABELCOLUMN].tolist()

    train_examples = [InputExample(texts=[s], label=l) for s, l in zip(train_sentences, train_labels)]

    def collate_fn(batch):
        texts = [example.texts[0] for example in batch]
        labels = torch.tensor([example.label for example in batch])
        return texts, labels

    train_dataloader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=batch_size,
        collate_fn=collate_fn
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model_with_head.parameters(), lr=learning_rate)

    total_steps = len(train_dataloader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )

    checkpoint_base_dir = os.path.join(model_name_dir, 'checkpoints')
    os.makedirs(checkpoint_base_dir, exist_ok=True)

    loss_list = []
    for epoch in range(num_epochs):
        model_with_head.train()
        epoch_loss = 0

        for step, (texts, labels) in enumerate(train_dataloader):
            labels = labels.to(device)
            optimizer.zero_grad()

            features = model.encode(texts, convert_to_tensor=True, device=device)
            features_dict = {'sentence_embedding': features}
            logits = classification_head(features_dict)

            loss = criterion(logits, labels)
            epoch_loss += loss.item()

            loss.backward()
            optimizer.step()
            scheduler.step()

        avg_loss = epoch_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{num_epochs} completed, Average Loss: {avg_loss:.4f}")
        loss_list.append(avg_loss)

        if (epoch + 1) % 1 == 0:
            checkpoint_dir = os.path.join(checkpoint_base_dir, f'epoch-{epoch+1}')
            os.makedirs(checkpoint_dir, exist_ok=True)
            model.save(checkpoint_dir)
            torch.save(classification_head.state_dict(), os.path.join(checkpoint_dir, "classification_head.pt"))
            print(f"Model checkpoint saved to {checkpoint_dir}")

    final_dir = os.path.join(model_name_dir, 'final')
    os.makedirs(final_dir, exist_ok=True)
    model.save(final_dir)
    torch.save(classification_head.state_dict(), os.path.join(final_dir, "classification_head.pt"))
    print(f"Final model saved to {final_dir}")

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, num_epochs+1), loss_list, marker='o')
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)

    plot_path = os.path.join(model_name_dir, 'training_loss.png')
    plt.savefig(plot_path)
    print(f"Training loss plot saved as '{plot_path}'")

    return model, classification_head

In [ ]:
def calculate_metrics(model, classification_head, dataset):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    classification_head.to(device)

    def evaluate_split(split):
        sentences = dataset[split][DATACOLUMN].tolist()

        if dataset[split][LABELCOLUMN].dtype == 'object':
            labels = [LABEL_MAP[label] for label in dataset[split][LABELCOLUMN]]
        else:
            labels = dataset[split][LABELCOLUMN].tolist()

        true_labels = np.array(labels)

        model.eval()
        classification_head.eval()
        all_preds = []

        batch_size = 32
        for i in range(0, len(sentences), batch_size):
            batch_texts = sentences[i:i+batch_size]

            with torch.no_grad():
                features = model.encode(batch_texts, convert_to_tensor=True, device=device)
                features_dict = {'sentence_embedding': features}

                logits = classification_head(features_dict)
                predictions = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(predictions)

        return np.array(all_preds), true_labels

    print("\n=== Validation Set Metrics ===")
    val_preds, val_labels = evaluate_split('val')

    val_accuracy = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average='weighted')

    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Validation F1 Score: {val_f1:.4f}")

    print("\n=== Test Set Metrics ===")
    test_preds, test_labels = evaluate_split('test')

    test_accuracy = accuracy_score(test_labels, test_preds)
    test_f1 = f1_score(test_labels, test_preds, average='weighted')

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test F1 Score: {test_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(test_labels, test_preds, target_names=list(EMOTION_INDEX.values())))

    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(EMOTION_INDEX.values()),
                yticklabels=list(EMOTION_INDEX.values()))
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig('../results/confusion_matrix.png')
    print("Confusion matrix saved as 'confusion_matrix.png'")

    return {
        'val_accuracy': val_accuracy,
        'val_f1': val_f1,
        'test_accuracy': test_accuracy,
        'test_f1': test_f1,
        'val_preds': val_preds,
        'val_labels': val_labels,
        'test_preds': test_preds,
        'test_labels': test_labels
    }

In [ ]:
dataset= load_dataset(clean=True)

In [ ]:
model, classification_head = train(
        dataset,
        num_epochs=3,
        batch_size=16,
        learning_rate=2e-5,
        model_name='all-MiniLM-L6-v2'
    )

In [ ]:
metrics = calculate_metrics(model, classification_head, dataset)

In [ ]:
metrics_3_epochs = calculate_metrics(model, classification_head, dataset)

In [ ]:
test_df = dataset['test'].copy()
test_df['predicted_label_sentenceTransformer'] = metrics_3_epochs['test_preds']
test_df['true_label'] = metrics_3_epochs['test_labels']

In [ ]:
simplified_df = test_df[['text', 'predicted_label_sentenceTransformer', 'true_label']]
simplified_df.to_csv("../results/all-MiniLM-L6-v2_predictions.csv", index=False)

RoBERTa Model

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

In [ ]:
model_name_roberta = "roberta-base"
roberta_tokenizer =  RobertaTokenizer.from_pretrained(model_name_roberta)

num_labels = 3
roberta_model = RobertaForSequenceClassification.from_pretrained(model_name_roberta, num_labels=num_labels).to(device)

In [ ]:
def tokenize_function(examples):
    return roberta_tokenizer(examples['text'], padding="max_length", truncation=True)

In [ ]:
def create_batch_tokenized_datasets_roberta(max_length=512, model_name="roberta-base"):
    dataframes = load_dataset(clean=True)

    hf_datasets = {}
    for split, df in dataframes.items():
        df = df.copy()
        df['label'] = df['my_label'].map(LABEL_MAP)
        hf_datasets[split] = HFDataset.from_pandas(df)

    dataset_dict = DatasetDict(hf_datasets)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            max_length=max_length,
            truncation=True,
            return_tensors="pt"
        )

    dataset_encoded = dataset_dict.map(tokenize, batched=True, batch_size=None)

    dataset_encoded.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )

    print(f"Created batch-tokenized datasets:")
    for split, dataset in dataset_encoded.items():
        print(f"  {split}: {len(dataset)} examples")

    return dataset_encoded

In [ ]:
dataset_encoded_roberta = create_batch_tokenized_datasets_roberta(max_length=512, model_name="roberta-base")

In [ ]:
dataset_encoded_roberta.set_format(type=None)

class CustomTorchDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(item['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(item['label'], dtype=torch.long)
        }


train_dataset = CustomTorchDataset(dataset_encoded_roberta["train"])
val_dataset = CustomTorchDataset(dataset_encoded_roberta["val"])

In [ ]:
logging_steps = len(dataset["train"]) // batch_size
save_steps = logging_steps
eval_steps = logging_steps

training_args = TrainingArguments(
    output_dir='../models/resultsroberta',
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    metric_for_best_model="f1",
    disable_tqdm=False,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    logging_steps=10,
    report_to="none"
)

In [ ]:
trainer_roberta = Trainer(model=roberta_model, args=training_args,
                  compute_metrics=compute_metrics,
                  train_dataset=dataset_encoded_roberta["train"],
                  eval_dataset=dataset_encoded_roberta["val"])
trainer_roberta.train();

In [ ]:
preds_output_roberta = trainer_roberta.predict(dataset_encoded_roberta["test"])
preds_output_roberta.metrics

In [ ]:
y_valid_roberta = np.array(dataset_encoded["test"]["label"])
y_preds_roberta = np.argmax(preds_output_roberta.predictions, axis=1)
labels = ['neutral', 'moderate', 'high-risk']
cm = confusion_matrix(y_valid_roberta, y_preds_roberta)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix RoBERTa')
plt.show()

In [ ]:
print('\tClassification Report for RoBERTa:\n\n',classification_report(y_valid_roberta,y_preds_roberta, target_names = ['neutral', 'moderate', 'high-risk']))

In [ ]:
df_preds_roberta = pd.DataFrame({
    "text": dataset_encoded["test"]["text"],
    "true_label": y_valid_roberta,
    "predicted_label_roberta": y_preds_roberta

})

In [ ]:
df_preds_roberta.to_csv("../results/roberta_predictions.csv", index=False)